In [1]:
import torch
import ultralytics

print("--- SYSTEM CHECK ---")
print(f"YOLO Version: {ultralytics.__version__}")
print(f"GPU Detected: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "WARNING: CPU ONLY")

--- SYSTEM CHECK ---
YOLO Version: 8.4.33
GPU Detected: NVIDIA GeForce RTX 4050 Laptop GPU


In [2]:
import json
import os
import yaml

# 1. Create the labels folder next to your images folder
os.makedirs('labels', exist_ok=True)

with open('dataset_train.json', 'r') as f:
    data = json.load(f)

# 2. Map IDs 1-79 to YOLO indices 0-78
sorted_cats = sorted(data['categories'], key=lambda x: x['id'])
class_map = {cat['id']: i for i, cat in enumerate(sorted_cats)}
names_list = [cat['name'] for cat in sorted_cats]
images_dict = {img['id']: img for img in data['images']}

print("📝 Generating YOLO labels...")
for ann in data['annotations']:
    img = images_dict.get(ann['image_id'])
    if not img: continue
    
    w, h = img['width'], img['height']
    bbox = ann['bbox'] # [x, y, width, height]
    
    # YOLO Math: Normalized coordinates
    x_c = (bbox[0] + bbox[2] / 2) / w
    y_c = (bbox[1] + bbox[3] / 2) / h
    w_n = bbox[2] / w
    h_n = bbox[3] / h
    
    # Filename matching (67cda248...png -> 67cda248...txt)
    file_id = os.path.splitext(os.path.basename(img['file_name']))[0]
    label_path = os.path.join('labels', f"{file_id}.txt")
    
    yolo_class = class_map[ann['category_id']]
    
    with open(label_path, 'a') as f:
        f.write(f"{yolo_class} {x_c:.6f} {y_c:.6f} {w_n:.6f} {h_n:.6f}\n")

print(f"✅ Created labels for your images. Ready for training!")

📝 Generating YOLO labels...
✅ Created labels for your images. Ready for training!


In [3]:
import json
import yaml

# 1. Open the 'map' to get the names
print("📖 Reading names from dataset_train.json...")
with open('dataset_train.json', 'r') as f:
    data = json.load(f)

# 2. Extract and sort the 79 categories perfectly (0 to 78)
categories = sorted(data['categories'], key=lambda x: x['id'])
names_list = [cat['name'] for cat in categories]

# 3. Define the GPS coordinates for YOLO
# We are using '.' because your images/labels are in the same folder
yaml_data = {
    'path': '.',           # Your project root
    'train': 'images',     # Where the .png files are
    'val': 'images',       # Using same folder for validation
    'nc': len(names_list), # Should be 79
    'names': names_list    # The full list of marine names
}

# 4. Save the file
with open('fathomnet.yaml', 'w') as f:
    yaml.dump(yaml_data, f, sort_keys=False)

print(f"✅ SUCCESS! Created 'fathomnet.yaml' with {len(names_list)} classes.")

📖 Reading names from dataset_train.json...
✅ SUCCESS! Created 'fathomnet.yaml' with 79 classes.


In [4]:
import json
import os

# 1. Create the labels folder
os.makedirs('labels', exist_ok=True)

with open('dataset_train.json', 'r') as f:
    data = json.load(f)

# 2. Map IDs 1-79 to YOLO 0-78
sorted_cats = sorted(data['categories'], key=lambda x: x['id'])
class_map = {cat['id']: i for i, cat in enumerate(sorted_cats)}
images_dict = {img['id']: img for img in data['images']}

print("📝 Writing YOLO labels to /labels/...")
for ann in data['annotations']:
    img = images_dict.get(ann['image_id'])
    if not img: continue
    
    w, h = img['width'], img['height']
    bbox = ann['bbox'] # [x, y, width, height]
    
    # YOLO Math: Center_X, Center_Y, Width, Height (Normalized 0.0 to 1.0)
    x_c = (bbox[0] + bbox[2] / 2) / w
    y_c = (bbox[1] + bbox[3] / 2) / h
    w_n = bbox[2] / w
    h_n = bbox[3] / h
    
    # Match the filename (e.g., 67cda248...png -> 67cda248...txt)
    file_id = os.path.splitext(os.path.basename(img['file_name']))[0]
    label_path = os.path.join('labels', f"{file_id}.txt")
    
    yolo_class = class_map[ann['category_id']]
    
    with open(label_path, 'a') as f:
        f.write(f"{yolo_class} {x_c:.6f} {y_c:.6f} {w_n:.6f} {h_n:.6f}\n")

print(f"✅ DONE! Labels generated. You are ready to train.")

📝 Writing YOLO labels to /labels/...
✅ DONE! Labels generated. You are ready to train.


In [ ]:
from ultralytics import YOLO
import torch

if __name__ == '__main__':
    # Initialize fresh YOLOv8 Nano model
    model = YOLO('yolov8n.pt') 

    print("🚀 RTX 4050: Initiating Training...")

    model.train(
        data='fathomnet.yaml', # Your new GPS file
        epochs=50,             # Full training cycle
        imgsz=640,             # Standard resolution
        batch=16,              # Good for 6GB VRAM
        device=0,              # Force GPU
        workers=0              # Windows Notebook safety setting
    )

🚀 RTX 4050: Initiating Training...
Ultralytics 8.4.33  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=fathomnet.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, over

In [1]:
from ultralytics import YOLO

if __name__ == '__main__':
    # 1. Load your "Save Game" instead of the blank model
    # Make sure 'train2' matches the folder name in your runs/detect/ directory!
    model = YOLO('runs/detect/train3/weights/last.pt') 

    print("🔄 Resuming training from the last saved epoch...")

    # 2. Tell YOLO to resume. You don't need to pass epochs, batch, or imgsz 
    # because it remembers all your settings from the last.pt file!
    model.train(resume=True)

🔄 Resuming training from the last saved epoch...
Ultralytics 8.4.33  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=fathomnet.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs\detect\train3\weights\last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train3, nbs=64, nms=False, opset=None

In [6]:
import json
import os
import shutil

# 1. Clear out the old misnamed labels
if os.path.exists('labels'):
    shutil.rmtree('labels')
os.makedirs('labels', exist_ok=True)

print("📖 Reading dataset_train.json...")
with open('dataset_train.json', 'r') as f:
    data = json.load(f)

# 2. Map IDs 1-79 to YOLO 0-78
sorted_cats = sorted(data['categories'], key=lambda x: x['id'])
class_map = {cat['id']: i for i, cat in enumerate(sorted_cats)}
images_dict = {img['id']: img for img in data['images']}

print("📝 Creating matching labels for Official Downloader filenames...")
count = 0
for ann in data['annotations']:
    img = images_dict.get(ann['image_id'])
    if not img: continue
    
    w, h = img['width'], img['height']
    bbox = ann['bbox'] # [x, y, width, height]
    
    # YOLO Math: Center_X, Center_Y, Width, Height (Normalized 0.0 to 1.0)
    x_c = (bbox[0] + bbox[2] / 2) / w
    y_c = (bbox[1] + bbox[3] / 2) / h
    w_n = bbox[2] / w
    h_n = bbox[3] / h
    
    # MATCHING THE OFFICIAL DOWNLOADER: Uses Image ID as filename
    label_path = os.path.join('labels', f"{ann['image_id']}.txt")
    
    yolo_class = class_map[ann['category_id']]
    
    with open(label_path, 'a') as f:
        f.write(f"{yolo_class} {x_c:.6f} {y_c:.6f} {w_n:.6f} {h_n:.6f}\n")
    count += 1

print(f"✅ SUCCESS! Created {count} labels using ID-based naming.")

📖 Reading dataset_train.json...
📝 Creating matching labels for Official Downloader filenames...
✅ SUCCESS! Created 23699 labels using ID-based naming.


In [7]:
import os
img_files = sorted(os.listdir('images'))[:5]
lbl_files = sorted(os.listdir('labels'))[:5]

print(f"Sample Images: {img_files}")
print(f"Sample Labels: {lbl_files}")

if img_files[0].split('.')[0] == lbl_files[0].split('.')[0]:
    print("💎 PERFECT MATCH! You are ready to train.")
else:
    print("❌ STILL MISMATCHED. Check the filenames above.")

Sample Images: ['1.png', '10.png', '100.png', '1000.png', '1001.png']
Sample Labels: ['1.txt', '10.txt', '100.txt', '1000.txt', '1001.txt']
💎 PERFECT MATCH! You are ready to train.


In [1]:
import json
import os
import requests
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm

# 1. Create a clean folder for the test exam
os.makedirs('test_images', exist_ok=True)

# 2. Read your test map
test_file = 'dataset_test.json' 

print(f"📖 Reading {test_file}...")
with open(test_file, 'r') as f:
    data = json.load(f)

images = data['images']
print(f"🌊 Found {len(images)} test images. Starting your RTX 4050 engine for download...")

# 3. High-speed download function
def download_test_image(img_data):
    file_name = os.path.basename(img_data['file_name'])
    url = img_data['coco_url']
    save_path = os.path.join('test_images', file_name)
    
    if not os.path.exists(save_path):
        try:
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                with open(save_path, 'wb') as f:
                    f.write(response.content)
        except:
            pass

# Using 20 workers to download fast
with ThreadPoolExecutor(max_workers=20) as executor:
    list(tqdm(executor.map(download_test_image, images), total=len(images)))

print("✅ Test images downloaded successfully! You are ready for the scan.")

📖 Reading dataset_test.json...
🌊 Found 325 test images. Starting your RTX 4050 engine for download...


  0%|          | 0/325 [00:00<?, ?it/s]

✅ Test images downloaded successfully! You are ready for the scan.


In [6]:
from ultralytics import YOLO

# Load your Peak Brain
model = YOLO('runs/detect/train3/weights/best.pt')

print("🚀 Re-scanning with Confidence Scores activated...")

# Run the scan and save the percentages!
results = model.predict(
    source='test_images/', 
    save=False,          # We don't need the drawn images again, saves time!
    save_txt=True,      
    save_conf=True,      # 🔴 CRUCIAL: Saves the percentage score
    conf=0.25,          
    name='test_exam_final' # Saving to a new, clean folder
)

print("✅ Scan complete. Ready for CSV conversion.")

🚀 Re-scanning with Confidence Scores activated...

image 1/325 c:\Users\azaed\OneDrive\Documents\FathomNet_Project\test_images\000b8e39-7240-49fd-9f50-713edcb28544.png: 384x640 1 Elpidia, 1 Peniagone, 22.5ms
image 2/325 c:\Users\azaed\OneDrive\Documents\FathomNet_Project\test_images\0058e081-ab45-4550-a667-5535ffa43ba5.png: 448x640 10 Funiculinas, 2 Umbellulas, 23.0ms
image 3/325 c:\Users\azaed\OneDrive\Documents\FathomNet_Project\test_images\006633fd-47a0-4e54-bec6-4f7d190b672b.png: 384x640 38 Paralomis multispinas, 19.0ms
image 4/325 c:\Users\azaed\OneDrive\Documents\FathomNet_Project\test_images\00b53944-ff7b-417b-ac60-01b1150d2e5f.png: 384x640 1 Peniagone, 20.0ms
image 5/325 c:\Users\azaed\OneDrive\Documents\FathomNet_Project\test_images\01ab37a8-2470-47e1-982a-f601c7f66c67.png: 384x640 (no detections), 18.0ms
image 6/325 c:\Users\azaed\OneDrive\Documents\FathomNet_Project\test_images\01b7012e-4a7d-4c57-a2d2-62f79debb0f5.png: 384x640 1 Apostichopus leukothele, 18.2ms
image 7/325 c:

In [20]:
import json
import os
import pandas as pd

print("🔍 Starting the Ultimate Matcher (with Concept Names)...")

# 1. Load the official test map
with open('dataset_test.json', 'r') as f:
    test_data = json.load(f)

# Create lookup dictionaries for images and category names
image_dict = {img['id']: img for img in test_data['images']}
categories = test_data.get('categories', [])

# Map YOLO IDs (0-78) to Official IDs AND to Scientific Names
sorted_cats = sorted(categories, key=lambda x: x['id'])
reverse_class_map = {i: cat['id'] for i, cat in enumerate(sorted_cats)}
id_to_name_map = {cat['id']: cat['name'] for cat in categories}

# IoU Math Function
def get_iou(box1, box2):
    x_left = max(box1[0], box2[0])
    y_top = max(box1[1], box2[1])
    x_right = min(box1[0] + box1[2], box2[0] + box2[2])
    y_bottom = min(box1[1] + box1[3], box2[1] + box2[3])
    if x_right < x_left or y_bottom < y_top: return 0.0
    intersection = (x_right - x_left) * (y_bottom - y_top)
    return intersection / float(box1[2] * box1[3] + box2[2] * box2[3] - intersection)

# 2. Read your YOLO predictions
labels_folder = 'runs/detect/test_exam_final/labels'
yolo_predictions = {} 

if os.path.exists(labels_folder):
    for txt_file in os.listdir(labels_folder):
        if not txt_file.endswith('.txt'): continue
        file_id = os.path.splitext(txt_file)[0]
        yolo_predictions[file_id] = []
        
        img_meta = next((img for img in test_data['images'] if os.path.splitext(os.path.basename(img['file_name']))[0] == file_id), None)
        if not img_meta: continue
        w_img, h_img = img_meta['width'], img_meta['height']
        
        with open(os.path.join(labels_folder, txt_file), 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) == 6:
                    yolo_class = int(parts[0])
                    x_c, y_c, w_n, h_n, conf = map(float, parts[1:])
                    # YOLO to Absolute Pixels
                    w_p, h_p = w_n * w_img, h_n * h_img
                    x_p, y_p = (x_c * w_img) - (w_p / 2), (y_c * h_img) - (h_p / 2)
                    yolo_predictions[file_id].append({
                        'category_id': reverse_class_map[yolo_class],
                        'confidence': conf,
                        'bbox': [x_p, y_p, w_p, h_p]
                    })

# 3. Match and generate the 788 rows
csv_data = []
official_annotations = test_data.get('annotations', [])

for ann in official_annotations:
    img_id = ann['image_id']
    official_bbox = ann['bbox']
    img_meta = image_dict.get(img_id)
    if not img_meta: continue
    file_id = os.path.splitext(os.path.basename(img_meta['file_name']))[0]
    
    best_cat_id = 1 # Fallback ID
    best_conf = 0.0
    best_iou = 0.0
    
    if file_id in yolo_predictions:
        for yolo_box in yolo_predictions[file_id]:
            iou = get_iou(official_bbox, yolo_box['bbox'])
            if iou > best_iou:
                best_iou, best_cat_id, best_conf = iou, yolo_box['category_id'], yolo_box['confidence']
    
    # Get the scientific name from the ID
    concept_name = id_to_name_map.get(best_cat_id, "Unknown")
    
    csv_data.append({
        'annotation_id': ann['id'],
        'image_id': img_id,
        'category_id': best_cat_id,
        'concept_name': concept_name, # 🔴 THE NEW COLUMN!
        'confidence': round(best_conf if best_iou > 0 else 0.0, 4),
        'bbox': f"[{round(official_bbox[0],2)}, {round(official_bbox[1],2)}, {round(official_bbox[2],2)}, {round(official_bbox[3],2)}]"
    })

# 4. Save the file
df = pd.DataFrame(csv_data)
# Ensure the column order is exactly as expected
df = df[['annotation_id', 'image_id', 'category_id', 'concept_name', 'confidence', 'bbox']]
df.to_csv('submission.csv', index=False)

print(f"🏆 DONE! 'submission.csv' now has {len(df)} rows and the 'concept_name' column.")

🔍 Starting the Ultimate Matcher (with Concept Names)...
🏆 DONE! 'submission.csv' now has 788 rows and the 'concept_name' column.


In [22]:
from ultralytics import YOLO

if __name__ == '__main__':
    # 1. Load the Medium model 'brain'
    model = YOLO('yolov8m.pt') 

    print("🚀 RTX 4050: Initiating Professional Training (Medium Model)...")

    # 2. Training with 'Taxonomic Focus'
    model.train(
        data='fathomnet.yaml',
        epochs=100,           # More time to learn the 79 species
        imgsz=640,            # High resolution for fine details
        batch=8,              # Safe for 6GB VRAM (We could try 12, but 8 is stable)
        patience=20,          # Stop early if the model stops learning to save your GPU
        optimizer='AdamW',    # Better for complex classification
        lr0=0.001,            # Standard starting learning rate
        cos_lr=True,          # Smoothly lowers learning rate for 'fine-tuning' at the end
        # --- Heavy Augmentation (Helps with murky water) ---
        fliplr=0.5,           # Helps AI recognize fish from both sides
        mosaic=1.0,           # Essential for finding small creatures
        name='FathomNet_Medium_Run'
    )

🚀 RTX 4050: Initiating Professional Training (Medium Model)...
Ultralytics 8.4.33  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=fathomnet.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=FathomNet_Medium_Run, nbs=64, nms=False, ops

In [23]:
from ultralytics import YOLO

# 1. Load the NEW 'Best' weights from your Medium run
model = YOLO('runs/detect/FathomNet_Medium_Run/weights/best.pt')

print("🚀 Scanning test images with the Medium model...")

# 2. Run the prediction
results = model.predict(
    source='test_images/', 
    save=False,          # No need to save images, saves space
    save_txt=True,      
    save_conf=True,      # Required for the matcher
    conf=0.10,           # We use a lower threshold here so the matcher has more options to pick from
    name='test_medium_final' 
)

print("✅ Scan complete! Labels are saved in runs/detect/test_medium_final/labels")

🚀 Scanning test images with the Medium model...

image 1/325 c:\Users\azaed\OneDrive\Documents\FathomNet_Project\test_images\000b8e39-7240-49fd-9f50-713edcb28544.png: 384x640 1 Elpidia, 1 Peniagone, 1 Scotoplanes globosa, 453.9ms
image 2/325 c:\Users\azaed\OneDrive\Documents\FathomNet_Project\test_images\0058e081-ab45-4550-a667-5535ffa43ba5.png: 448x640 29 Funiculinas, 1 Sebastolobus, 13 Umbellulas, 198.6ms
image 3/325 c:\Users\azaed\OneDrive\Documents\FathomNet_Project\test_images\006633fd-47a0-4e54-bec6-4f7d190b672b.png: 384x640 38 Paralomis multispinas, 90.1ms
image 4/325 c:\Users\azaed\OneDrive\Documents\FathomNet_Project\test_images\00b53944-ff7b-417b-ac60-01b1150d2e5f.png: 384x640 1 Holothuroidea, 1 Peniagone, 88.6ms
image 5/325 c:\Users\azaed\OneDrive\Documents\FathomNet_Project\test_images\01ab37a8-2470-47e1-982a-f601c7f66c67.png: 384x640 1 Benthocodon pedunculata, 86.0ms
image 6/325 c:\Users\azaed\OneDrive\Documents\FathomNet_Project\test_images\01b7012e-4a7d-4c57-a2d2-62f79de

In [24]:
import json
import os
import pandas as pd

# 1. Load the official test map
with open('dataset_test.json', 'r') as f:
    test_data = json.load(f)

image_dict = {img['id']: img for img in test_data['images']}
categories = test_data.get('categories', [])
sorted_cats = sorted(categories, key=lambda x: x['id'])
reverse_class_map = {i: cat['id'] for i, cat in enumerate(sorted_cats)}
id_to_name_map = {cat['id']: cat['name'] for cat in categories}

def get_iou(box1, box2):
    x_left, y_top = max(box1[0], box2[0]), max(box1[1], box2[1])
    x_right, y_bottom = min(box1[0] + box1[2], box2[0] + box2[2]), min(box1[1] + box1[3], box2[1] + box2[3])
    if x_right < x_left or y_bottom < y_top: return 0.0
    intersection = (x_right - x_left) * (y_bottom - y_top)
    return intersection / float(box1[2] * box1[3] + box2[2] * box2[3] - intersection)

# 2. Read your NEW Medium predictions
labels_folder = 'runs/detect/test_medium_final/labels'
yolo_predictions = {} 

if os.path.exists(labels_folder):
    for txt_file in os.listdir(labels_folder):
        if not txt_file.endswith('.txt'): continue
        file_id = os.path.splitext(txt_file)[0]
        yolo_predictions[file_id] = []
        img_meta = next((img for img in test_data['images'] if os.path.splitext(os.path.basename(img['file_name']))[0] == file_id), None)
        if not img_meta: continue
        w_img, h_img = img_meta['width'], img_meta['height']
        with open(os.path.join(labels_folder, txt_file), 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) == 6:
                    y_class, x_c, y_c, w_n, h_n, conf = map(float, parts)
                    w_p, h_p = w_n * w_img, h_n * h_img
                    x_p, y_p = (x_c * w_img) - (w_p / 2), (y_c * h_img) - (h_p / 2)
                    yolo_predictions[file_id].append({'cat': reverse_class_map[int(y_class)], 'conf': conf, 'bbox': [x_p, y_p, w_p, h_p]})

# 3. Match to the 788 Official Rows
csv_data = []
for ann in test_data.get('annotations', []):
    img_meta = image_dict.get(ann['image_id'])
    file_id = os.path.splitext(os.path.basename(img_meta['file_name']))[0]
    best_cat_id, best_conf, best_iou = 1, 0.0, 0.0
    if file_id in yolo_predictions:
        for y_box in yolo_predictions[file_id]:
            iou = get_iou(ann['bbox'], y_box['bbox'])
            if iou > best_iou: best_iou, best_cat_id, best_conf = iou, y_box['cat'], y_box['conf']
    
    csv_data.append({
        'annotation_id': ann['id'], 'image_id': ann['image_id'], 'category_id': best_cat_id,
        'concept_name': id_to_name_map.get(best_cat_id, "Unknown"), 'confidence': round(best_conf, 4),
        'bbox': f"[{round(ann['bbox'][0],2)}, {round(ann['bbox'][1],2)}, {round(ann['bbox'][2],2)}, {round(ann['bbox'][3],2)}]"
    })

df = pd.DataFrame(csv_data)
df[['annotation_id', 'image_id', 'category_id', 'concept_name', 'confidence', 'bbox']].to_csv('submission_medium.csv', index=False)
print("🏆 DONE! Your 'submission_medium.csv' is ready.")

🏆 DONE! Your 'submission_medium.csv' is ready.


In [27]:
import json
import os
import cv2
import random
from tqdm import tqdm

# 1. Load your training data
with open('dataset_train.json', 'r') as f:
    data = json.load(f)

# 2. Setup folder structure
for split in ['train', 'val']:
    os.makedirs(f'classifier_data/{split}', exist_ok=True)

# Create a lookup using IDs since your folder has '1.png', '2.png', etc.
# We will try both .png and .jpg just in case
image_ids = [img['id'] for img in data['images']]

print("✂️ Cropping fish and splitting into Train/Val sets...")
random.seed(42)

# Stats counter
missing_files = 0
success_count = 0

for ann in tqdm(data['annotations']):
    cat_id = ann['category_id']
    img_id = ann['image_id']
    bbox = ann['bbox'] 
    
    # Check for the numbered filename (e.g., '1.png')
    img_filename = f"{img_id}.png"
    img_path = os.path.join('images', img_filename)
    
    # If .png isn't found, try .jpg
    if not os.path.exists(img_path):
        img_path = os.path.join('images', f"{img_id}.jpg")

    if not os.path.exists(img_path):
        missing_files += 1
        continue # Skip to the next one if the file is truly missing
    
    img = cv2.imread(img_path)
    
    if img is not None:
        split = 'train' if random.random() < 0.8 else 'val'
        x, y, w, h = map(int, bbox)
        
        # Ensure coordinates stay inside the image
        y_max, x_max = img.shape[:2]
        crop = img[max(0,y):min(y+h, y_max), max(0,x):min(x+w, x_max)]
        
        if crop.size > 0:
            save_path = f"classifier_data/{split}/{cat_id}"
            os.makedirs(save_path, exist_ok=True)
            cv2.imwrite(f"{save_path}/{ann['id']}.jpg", crop)
            success_count += 1

print(f"\n✅ Finished!")
print(f"📦 Successfully cropped: {success_count} images")
print(f"⚠️ Skipped (not found): {missing_files} annotations")

✂️ Cropping fish and splitting into Train/Val sets...


100%|██████████| 23699/23699 [18:55<00:00, 20.87it/s] 


✅ Finished!
📦 Successfully cropped: 23698 images
⚠️ Skipped (not found): 0 annotations


In [1]:
from ultralytics import YOLO

# 1. Load the Medium Classification model
model = YOLO('yolov8m-cls.pt')

print("🎓 Training the Specialist Classifier on 23,698 crops...")

# 2. Train the model
# We use imgsz=224 because it's the standard for classification,
# allowing for a high batch size and very fast training.
results = model.train(
    data='classifier_data',
    epochs=50,
    imgsz=224,
    batch=32,
    ##workers=4,
    optimizer='AdamW',
    name='FathomNet_Specialist_v1'
)

🎓 Training the Specialist Classifier on 23,698 crops...
Ultralytics 8.4.33  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=classifier_data, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=FathomNet_Specialist_v14, nbs=64, nms=False, o

In [2]:
import json
import os
import cv2
import pandas as pd
from ultralytics import YOLO
from tqdm import tqdm

# 1. Load your NEW Specialist model
# Note: Using your specific path from the logs
model_path = r'runs\classify\FathomNet_Specialist_v14\weights\best.pt'
model = YOLO(model_path)

# 2. Load the official test map and categories
with open('dataset_test.json', 'r') as f:
    test_data = json.load(f)

# Mapping folder indices to the actual category IDs
# Since folders were named after category IDs, model.names contains them as strings
idx_to_cat_id = {int(k): int(v) for k, v in model.names.items()}
id_to_name = {cat['id']: cat['name'] for cat in test_data['categories']}

submission_data = []

print("🧪 Specialist is identifying the 788 official test boxes...")

for ann in tqdm(test_data['annotations']):
    # Find the corresponding image
    img_meta = next(img for img in test_data['images'] if img['id'] == ann['image_id'])
    img_path = os.path.join('test_images', img_meta['file_name'])
    img = cv2.imread(img_path)
    
    if img is not None:
        # Crop the official box [x, y, w, h]
        x, y, w, h = map(int, ann['bbox'])
        h_img, w_img = img.shape[:2]
        
        # Ensure crop stays within image boundaries
        crop = img[max(0, y):min(y+h, h_img), max(0, x):min(x+w, w_img)]
        
        if crop.size > 0:
            # 3. Predict with the Specialist
            results = model.predict(crop, verbose=False)[0]
            top_idx = results.probs.top1
            pred_cat_id = idx_to_cat_id[top_idx]
            conf = float(results.probs.top1conf)
            
            submission_data.append({
                'annotation_id': ann['id'],
                'image_id': ann['image_id'],
                'category_id': pred_cat_id,
                'concept_name': id_to_name.get(pred_cat_id, "Unknown"),
                'confidence': round(conf, 4),
                'bbox': f"[{ann['bbox'][0]}, {ann['bbox'][1]}, {ann['bbox'][2]}, {ann['bbox'][3]}]"
            })

# 4. Save the high-accuracy submission
df = pd.DataFrame(submission_data)
output_name = 'submission_specialist_V14.csv'
df.to_csv(output_name, index=False)

print(f"🏆 DONE! Your Specialist submission is ready: {output_name}")

🧪 Specialist is identifying the 788 official test boxes...


100%|██████████| 788/788 [00:47<00:00, 16.55it/s]

🏆 DONE! Your Specialist submission is ready: submission_specialist_V14.csv


In [3]:
import torch
from ultralytics import YOLO

# 1. Clear the GPU memory
torch.cuda.empty_cache()

# 2. Load the LARGE Classification model
model = YOLO('yolov8l-cls.pt') 

print("🚀 Starting High-Res Specialist Training (448px)...")

# 3. Training with 'Fine-Detail' settings
results = model.train(
    data='classifier_data',
    epochs=50,
    imgsz=448,             # 🔴 2x higher resolution for taxonomic details
    batch=8,               # ⚠️ Lowered to fit 6GB VRAM
    patience=15,           # Stop if it doesn't improve for 15 epochs
    optimizer='AdamW',
    name='FathomNet_Large_448',
    amp=True               # Essential for 6GB cards (saves memory)
)

🚀 Starting High-Res Specialist Training (448px)...
Ultralytics 8.4.33  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=classifier_data, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=448, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8l-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=FathomNet_Large_448, nbs=64, nms=False, opset=None, 

In [4]:
import json
import os
import cv2
import pandas as pd
from ultralytics import YOLO
from tqdm import tqdm
import numpy as np

# 1. Load your model (Use your 84% accuracy model for best results)
model = YOLO(r'runs\classify\FathomNet_Specialist_v14\weights\best.pt')

with open('dataset_test.json', 'r') as f:
    test_data = json.load(f)

idx_to_cat_id = {int(k): int(v) for k, v in model.names.items()}
id_to_name = {cat['id']: cat['name'] for cat in test_data['categories']}

submission_data = []

print("🧪 Running Taxonomy-Aware Inference...")

for ann in tqdm(test_data['annotations']):
    img_meta = next(img for img in test_data['images'] if img['id'] == ann['image_id'])
    img_path = os.path.join('test_images', img_meta['file_name'])
    img = cv2.imread(img_path)
    
    if img is not None:
        x, y, w, h = map(int, ann['bbox'])
        h_img, w_img = img.shape[:2]
        crop = img[max(0, y):min(y+h, h_img), max(0, x):min(x+w, w_img)]
        
        if crop.size > 0:
            results = model.predict(crop, verbose=False)[0]
            
            # --- THE HACK: Look at Top-5 instead of just Top-1 ---
            top5_indices = results.probs.top5
            top5_confidences = results.probs.top5conf.cpu().numpy()
            
            # For now, we take the Top-1, but we 'filter' by confidence.
            # If the Top-1 confidence is low (e.g., < 0.4), the AI is guessing.
            # In a true winner's script, you would check if the Top-5 
            # share the same 'Family' and pick the most common Family.
            
            best_idx = top5_indices[0]
            pred_cat_id = idx_to_cat_id[best_idx]
            conf = float(top5_confidences[0])
            
            submission_data.append({
                'annotation_id': ann['id'],
                'image_id': ann['image_id'],
                'category_id': pred_cat_id,
                'concept_name': id_to_name.get(pred_cat_id, "Unknown"),
                'confidence': round(conf, 4),
                'bbox': f"[{ann['bbox'][0]}, {ann['bbox'][1]}, {ann['bbox'][2]}, {ann['bbox'][3]}]"
            })

df = pd.DataFrame(submission_data)
df.to_csv('submission_taxonomy_hack.csv', index=False)
print("🏆 Upload 'submission_taxonomy_hack.csv' to Kaggle!")

🧪 Running Taxonomy-Aware Inference...


100%|██████████| 788/788 [00:56<00:00, 13.93it/s]

🏆 Upload 'submission_taxonomy_hack.csv' to Kaggle!


In [6]:
import json
import os
import cv2
import pandas as pd
import torch
from ultralytics import YOLO
from tqdm import tqdm

# 1. Load BOTH models
model_med = YOLO(r'runs\classify\FathomNet_Specialist_v14\weights\best.pt')  # 84% acc
model_lge = YOLO(r'runs\classify\FathomNet_Large_448\weights\best.pt')       # 68% acc

# 2. Load the official test map
with open('dataset_test.json', 'r') as f:
    test_data = json.load(f)

# Ensure both models use the same class mapping
idx_to_cat_id = {int(k): int(v) for k, v in model_med.names.items()}
id_to_name = {cat['id']: cat['name'] for cat in test_data['categories']}

ensemble_results = []

print("🧠 Both 'brains' are analyzing the test boxes... this will take a moment.")

for ann in tqdm(test_data['annotations']):
    img_meta = next(img for img in test_data['images'] if img['id'] == ann['image_id'])
    img_path = os.path.join('test_images', img_meta['file_name'])
    img = cv2.imread(img_path)
    
    if img is not None:
        x, y, w, h = map(int, ann['bbox'])
        h_img, w_img = img.shape[:2]
        crop = img[max(0, y):min(y+h, h_img), max(0, x):min(x+w, w_img)]
        
        if crop.size > 0:
            # Brain 1 (Medium) - Look at 224px
            res_med = model_med.predict(crop, imgsz=224, verbose=False)[0]
            probs_med = res_med.probs.data.cpu().numpy() # Probability list for all 79 classes
            
            # Brain 2 (Large) - Look at 448px
            res_lge = model_lge.predict(crop, imgsz=448, verbose=False)[0]
            probs_lge = res_lge.probs.data.cpu().numpy()
            
            # --- THE ENSEMBLE CALCULATION ---
            # We give the 84% model 70% weight and the 68% model 30% weight
            combined_probs = (0.7 * probs_med) + (0.3 * probs_lge)
            
            # Pick the final winner
            best_idx = combined_probs.argmax()
            pred_cat_id = idx_to_cat_id[best_idx]
            final_conf = combined_probs[best_idx]
            
            ensemble_results.append({
                'annotation_id': ann['id'],
                'image_id': ann['image_id'],
                'category_id': pred_cat_id,
                'concept_name': id_to_name.get(pred_cat_id, "Unknown"),
                'confidence': round(float(final_conf), 4),
                'bbox': f"[{ann['bbox'][0]}, {ann['bbox'][1]}, {ann['bbox'][2]}, {ann['bbox'][3]}]"
            })

# 3. Save as a NEW file (doesn't delete your old ones)
df = pd.DataFrame(ensemble_results)
df.to_csv('submission_ensemble_M_L.csv', index=False)
print("🏆 Ensemble Complete! Try uploading 'submission_ensemble_M_L.csv' to Kaggle.")

🧠 Both 'brains' are analyzing the test boxes... this will take a moment.


100%|██████████| 788/788 [01:52<00:00,  6.99it/s]

🏆 Ensemble Complete! Try uploading 'submission_ensemble_M_L.csv' to Kaggle.
